## 0.1 Prepare dataset

In [ ]:
import requests
import tarfile
from pathlib import Path
import shutil
import tempfile

# Dataset URL
dataset_url = "http://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz"

# Use path for temporary files, then copy to data directory
tmp_dir = Path(tempfile.gettempdir())
dataset_file = tmp_dir / "booksummaries.tar.gz"

# Path to data directory
data_dir = Path.cwd()
target_file = data_dir / "booksummaries.txt"

if not target_file.exists():
    print(f"Downloading CMU Book Summary Dataset from {dataset_url}...")
    
    # Download the file to /tmp
    response = requests.get(dataset_url, stream=True)
    response.raise_for_status()
    
    dataset_file.write_bytes(response.content)
    print(f"Downloaded to {dataset_file}")
    
    # Extract the archive
    print("Extracting archive...")
    with tarfile.open(dataset_file, "r:gz") as tar:
        tar.extractall(path=tmp_dir, filter='data')
    
    # Copy file to data directory
    extracted_file = tmp_dir / "booksummaries" / "booksummaries.txt"
    shutil.copy(str(extracted_file), str(target_file))
    
    # Cleanup
    shutil.rmtree(tmp_dir / "booksummaries")
    dataset_file.unlink()
    
    print(f"✓ Dataset ready: {target_file}")
else:
    print(f"✓ Dataset already exists: {target_file}")

## 1. Imports

In [ ]:
!pip install bertopic
!pip install bertopic[visualization]
!pip install pandas scikit-learn nltk

In [ ]:
import pandas as pd
from bertopic import BERTopic
import re
import nltk
from sklearn.feature_extraction.text import CountVectorizer
import os

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

## 2. Loading and preliminary data cleaning

We load the dataset and remove rows that do not contain plot summaries. We also drop duplicate book titles to ensure each book is unique.

In [30]:
file_path = 'booksummaries.txt'

if not os.path.exists(file_path):
    print(f"File {file_path} not foud.")
else:
    column_names = ['wiki_id', 'freebase_id', 'book_title', 'author', 'publication_date', 'genres', 'plot_summary']
    df = pd.read_csv(file_path, sep='\t', header=None, names=column_names)

    df.dropna(subset=['plot_summary'], inplace=True)
    df.drop_duplicates(subset=['book_title'], inplace=True)
    
    # Taking a sample of 3000 books for faster processing
    # df_sample = df.sample(n=3000, random_state=42)
    df_sample = df.copy()

    print(f"Loaded and processed {len(df_sample)} books.")
    print("Sample Data:")
    display(df_sample.head())

Loaded and processed 16277 books.
Sample Data:


,wiki_id,freebase_id,book_title,author,publication_date,genres,plot_summary
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,986,/m/0ldx,The Plague,Albert Camus,1947,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...
3,1756,/m/0sww,An Enquiry Concerning Human Understanding,David Hume,NaN,NaN,The argument of the Enquiry proceeds by a ser...
4,2080,/m/0wkt,A Fire Upon the Deep,Vernor Vinge,NaN,"{""/m/03lrw"": ""Hard science fiction"", ""/m/06n90...",The novel posits that space around the Milky ...


## 3. Text Preprocessing

We create a function to clean the plot summaries: we remove special characters, stop-words (commonly occurring words like 'the' and 'a') and perform lemmatization (reducing words to their base form).

In [31]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Removing special characters and digits
    text = re.sub(r'\W|\d', ' ', text)

    text = text.lower()

    tokens = nltk.word_tokenize(text)

    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)

if 'df_sample' in locals():
    print("Beginning text processing...")
    df_sample['processed_summary'] = df_sample['plot_summary'].apply(preprocess_text)
    print("Text processing finished.")

    print("\n--- Example --- ")
    print("Original summary:")
    print(df_sample.iloc[0]['plot_summary'][:500])
    print("\nSummary after processing:")
    print(df_sample.iloc[0]['processed_summary'][:500])

Beginning text processing...
Text processing finished.

--- Example --- 
Original summary:
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philosophy. The animals revolt and drive the drunken and irresponsible Mr Jones from the farm, renaming it "Animal Farm". They adopt Seven Commandments of Animal-ism, the most important of which is, "All a

Summary after processing:
old major old boar manor farm call animal farm meeting compare human parasite teach animal revolutionary song beast england major dy two young pig snowball napoleon assume command turn dream philosophy animal revolt drive drunken irresponsible jones farm renaming animal farm adopt seven commandment animal ism important animal equal snowball attempt teach animal reading writing f

## 4. Training BERTopic

BERTopic will automatically extract topics from the processed summaries. `CountVectorizer` is used to remove English stop-words during the creation of the topic representation.

In [32]:
if 'df_sample' in locals():
    documents = df_sample['processed_summary'].tolist()

    vectorizer_model = CountVectorizer(stop_words="english")

    # min_topic_size - help to avoid very small topics
    topic_model = BERTopic(
        language="english", 
        vectorizer_model=vectorizer_model,
        min_topic_size=20,
        verbose=True
    )

    topics, probabilities = topic_model.fit_transform(documents)
    
    print("\nModel has been trained successfully!")
    print(f"Found {len(topic_model.get_topic_info())} topics.")

2025-10-19 17:54:03,669 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 509/509 [02:32<00:00,  3.33it/s]
2025-10-19 17:56:39,413 - BERTopic - Embedding - Completed ✓
2025-10-19 17:56:39,414 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-19 17:56:54,653 - BERTopic - Dimensionality - Completed ✓
2025-10-19 17:56:54,654 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-19 17:56:55,208 - BERTopic - Cluster - Completed ✓
2025-10-19 17:56:55,212 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-19 17:56:57,186 - BERTopic - Representation - Completed ✓



Model has been trained successfully!
Found 64 topics.


## 5. Visualization and results analysis

### 5.1. Most frequent topics

In [33]:
if 'topic_model' in locals():
    display(topic_model.get_topic_info())

,Topic,Count,Name,Representation,Representative_Docs
0,-1,9517,-1_life_time_book_story,"[life, time, book, story, year, father, new, m...",[max carver son watchmaker moved family city o...
1,0,1115,0_king_dragon_magic_power,"[king, dragon, magic, power, lord, army, city,...",[anguissette comtesse phèdre delaunay montrève...
2,1,583,1_murder_killer_detective_case,"[murder, killer, detective, case, police, crim...",[novel like skinner series set edinburgh featu...
3,2,489,2_novel_story_book_life,"[novel, story, book, life, character, relation...",[novel plot called plot female socialization h...
4,3,487,3_earth_planet_human_ship,"[earth, planet, human, ship, space, crew, alie...",[story begin year year future time novel writi...
...,...,...,...,...,...
59,58,21,58_poirot_mr_murder_hastings,"[poirot, mr, murder, hastings, joyce, norma, j...",[hercule poirot board orient express constanti...
60,59,21,59_sharpe_french_harper_leroux,"[sharpe, french, harper, leroux, hakeswill, br...",[novel begin richard sharpe second wife jane b...
61,60,21,60_climate_energy_carbon_warming,"[climate, energy, carbon, warming, global, emi...",[book consists three part epilogue drawing fre...
62,61,20,61_leamas_mundt_fiedler_german,"[leamas, mundt, fiedler, german, liz, agent, b...",[cia agent blackford oakes sent west berlin ea...


### 5.2. Visualization of topics as a bar chart

This chart shows the 10 most popular topics and the keywords that define them.

In [34]:
if 'topic_model' in locals():
    display(topic_model.visualize_barchart(top_n_topics=10))

### 5.3. Map of distances between topics

It shows topics as bubbles. The size of a bubble corresponds to its popularity. Bubbles located close to each other are thematically similar. It help to understand the relationships between topics.

In [35]:
if 'topic_model' in locals():
    display(topic_model.visualize_topics())

### 5.4. Hierarchical clustering of topics

Shows how topics can be grouped into larger clusters. Useful for identifying overarching themes in literature.

In [36]:
if 'topic_model' in locals():
    display(topic_model.visualize_hierarchy())

## 6. Saving the Model

We save the trained model for easy reuse later. We also save the processed data for use in an API.

In [37]:
if 'topic_model' in locals():
    model_dir = 'model'
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model_path = os.path.join(model_dir, 'cmu_books_bertopic_model')
    topic_model.save(model_path, serialization='safetensors')

    df_sample.to_csv('processed_book_data.csv', index=False)

    print(f"Model has been saved in: {model_path}")
    print("Precessed data has been saved in: processed_book_data.csv")

Model has been saved in: model\cmu_books_bertopic_model
Precessed data has been saved in: processed_book_data.csv
